In [0]:
from pyspark.sql.functions import col, unbase64, from_json
from pyspark.sql.types import StructType, StructField, DoubleType, ArrayType, StringType

# 1. Set your custom Unity Catalog path
volume_path = "/Volumes/opcuatest/telemetry_schema/raw_volume/open546.json"

# 2. Load the JSON Array from the volume
df_raw = spark.read.option("multiline", "true").json(volume_path)

# 3. Decode the Base64 encrypted string payload
df_decoded = df_raw.withColumn("DecodedText", unbase64(col("Body")).cast("string"))

# 4. Define the schema to map the open62541 variables
opcua_schema = StructType([
    StructField("Messages", ArrayType(StructType([
        StructField("Payload", StructType([
            StructField("Temperature", StructType([StructField("Value", DoubleType())])),
            StructField("Pressure", StructType([StructField("Value", DoubleType())])),
            StructField("Vibration", StructType([StructField("Value", DoubleType())]))
        ]))
    ])))
])

# 5. Extract variables into flat columns
df_final = df_decoded.withColumn("ParsedData", from_json(col("DecodedText"), opcua_schema)).select(
    col("id").alias("cosmos_id"),
    col("ParsedData.Messages")[0]["Payload"]["Temperature"]["Value"].alias("temperature"),
    col("ParsedData.Messages")[0]["Payload"]["Pressure"]["Value"].alias("pressure"),
    col("ParsedData.Messages")[0]["Payload"]["Vibration"]["Value"].alias("vibration")
)

# 6. Drop any row missing critical sensor metrics before processing
df_clean = df_final.dropna(subset=["temperature", "pressure", "vibration"])

# 7. Convert to Pandas for Scikit-Learn
pdf = df_clean.toPandas()
print(f"Successfully loaded and parsed {len(pdf)} telemetry records.")
display(pdf.head())

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# 1. Scale features so different sensor magnitudes don't bias the model
scaler = StandardScaler()
feature_cols = ['temperature', 'pressure', 'vibration']
pdf_scaled = scaler.fit_transform(pdf[feature_cols])

# 2. Initialize the Isolation Forest Model
# contamination=0.02 means we expect up to 2% of factory data points to be abnormal anomalies
model = IsolationForest(contamination=0.02, random_state=42, n_estimators=100)

# 3. Train the model and append predictions back to the data
model.fit(pdf_scaled)
pdf['anomaly_flag'] = model.predict(pdf_scaled) #  1 = Normal, -1 = Anomaly
pdf['anomaly_score'] = model.decision_function(pdf_scaled) # Lower/negative score means high anomaly severity

print("Model training complete.")

In [0]:
# Isolate rows where the flag is -1 (Anomalies)
flagged_anomalies = pdf[pdf['anomaly_flag'] == -1]

print(f"=== ANOMALY ANALYSIS REPORT ===")
print(f"Total Operational Records: {len(pdf)}")
print(f"Total Anomalies Detected: {len(flagged_anomalies)}")

# Display the critical records requiring engineering attention
display(flagged_anomalies[['cosmos_id', 'temperature', 'pressure', 'vibration', 'anomaly_score']])

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
colors = {1: '#1f77b4', -1: '#d62728'} # Blue for normal, Red for anomalies

plt.scatter(pdf['temperature'], pdf['pressure'], c=pdf['anomaly_flag'].map(colors), alpha=0.7, edgecolors='k')
plt.title('Factory Operational Integrity Monitor (Temperature vs Pressure)')
plt.xlabel('Temperature (°C)')
plt.ylabel('Pressure (bar)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()